# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames the refresh queue as a ranking problem. The model is not the product; the product is a prioritized list of pages a content editor should review first.

I am using the starter slice only to define the task honestly: one row is one pseudonymized content item for one client, and the output should help a human decide what to refresh, expand, or leave alone.

## 1. My lane as an ML task (type)

**Ranking / scoring.** I want an ordered refresh queue, not a binary yes/no answer. The model should assign a priority score so the highest-risk, highest-opportunity pages rise to the top of the editor's list.

This is a ranking problem because the decision is "which pages first?" and the action is to review the top K pages under limited editorial capacity.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

for candidate in [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Loaded {data_path}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Clients: {df['client_id'].nunique():,}")
print("One row = one pseudonymized content item for one client.")
display(df[["content_id", "client_id", "content_type", "main_intent", "impressions_90d", "avg_position"]].head(5))

Loaded ../../data/raw/content_refresh_anonymized.csv
Rows: 30,000
Columns: 44
Clients: 32
One row = one pseudonymized content item for one client.


,content_id,client_id,content_type,main_intent,impressions_90d,avg_position
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,10.6
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,20.3
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,36.5
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,44.0


## 2. Target or proxy

**Starter proxy:** a binary decline flag derived from `trend_direction` (`1` when the row is `down`, `0` otherwise). That label is a defined proxy built from an observed window, so I would treat it as a teaching target here rather than a final production label.

For the real project, I would replace the proxy with a future-window outcome measured after the score is produced. That keeps the model predictive instead of circular.

In [2]:
target_proxy = (df["trend_direction"] == "down").astype("int8")
target_frame = pd.DataFrame(
    {
        "trend_direction": df["trend_direction"],
        "target_is_declining": target_proxy,
    }
)

print("Proxy target sketch:")
print("1 = down, 0 = not down")
print(f"Declining share: {target_proxy.mean():.1%}")
display(target_frame.sample(8, random_state=42))

Proxy target sketch:
1 = down, 0 = not down
Declining share: 54.2%


,trend_direction,target_is_declining
2308,stable,0
22404,up,0
23397,new,0
25058,down,1
2664,down,1
8511,up,0
5148,stable,0
7790,down,1


## 3. Success metric

**Precision@50.** The output is a ranked queue, and the team can only act on a short list each week. Good means the top 50 pages contain a higher share of truly declining pages than a transparent baseline or random ordering.

I prefer this metric because it matches the workflow: editors do not inspect every page, they inspect the highest-priority ones first.

In [3]:
queue_size = 50
decline_base_rate = target_proxy.mean()

print(f"Weekly queue size: top {queue_size}")
print(f"Starter proxy base rate: {decline_base_rate:.1%}")
print("Good = precision@50 materially above the base rate and a simple rule baseline.")

Weekly queue size: top 50
Starter proxy base rate: 54.2%
Good = precision@50 materially above the base rate and a simple rule baseline.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item for one client.** The dataframe below is the slice I would actually rank: content-level rows with the fields I need to decide which pages deserve editorial attention first.

The target sketch is also shown here so the unit of analysis and the label are visible together.

In [4]:
lane_df = df[[
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "days_since_last_update",
    "trend_direction",
]].copy()

lane_df["target_is_declining"] = (lane_df["trend_direction"] == "down").astype("int8")

print(f"Lane dataframe shape: {lane_df.shape[0]:,} rows × {lane_df.shape[1]} columns")
display(lane_df.head(10))

Lane dataframe shape: 30,000 rows × 11 columns


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,avg_position,days_since_last_update,trend_direction,target_is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,10.6,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,20.3,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,36.5,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,6.2,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,44.0,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,8.5,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,7.0,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,21.2,22,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,68,46.0,20,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,3,4.9,104,down,1


## 5. Why ML beats a fixed rule here

A fixed rule like "refresh anything stale and visible" is too blunt. It catches only a tiny slice of the catalog, and the best pages are not defined by one threshold — they depend on traffic, freshness, position, intent, content length, and momentum together.

ML is useful here because the pattern is real but messy: the ranking needs to weigh several signals at once and sort thousands of pages into a short editorial queue.

In [5]:
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
page_one_old = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
thin_visible = (df["word_count"].fillna(0) > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)

print(f"Stale + visible rule matches: {stale_visible.sum():,} pages")
print(f"Page 1 + old rule matches:    {page_one_old.sum():,} pages")
print(f"Thin + visible rule matches:   {thin_visible.sum():,} pages")
print("A single if-statement is too narrow for a queue this large.")

Stale + visible rule matches: 17 pages
Page 1 + old rule matches:    7,076 pages
Thin + visible rule matches:   82 pages
A single if-statement is too narrow for a queue this large.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.